# Route choice with a random utility model
**Input:** route attributes, taste parameters  
**Tool:** Python, numpy

Random utility: traveler *n* gets utility $U_{nj} = V_j + \varepsilon_{nj}$ from route *j* and picks the route with the highest $U_{nj}$. With i.i.d. Gumbel errors this yields the multinomial logit probabilities $P_j = e^{V_j}/\sum_k e^{V_k}$.

### Step 0. Import packages

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)   # fixed seed so results are reproducible

### Step 1. Input the choice set (times in minutes, slide 27)

In [2]:
choice_set = pd.DataFrame({
    'choice':          [1, 2, 3],
    'route':           ['U6-U3', 'U4-U1', 'U4 + walk'],
    'walking_time':    [1, 1, 10],
    'in_transit_time': [11, 9, 7],
    'transfer_time':   [1, 1, 0]})
choice_set['total_time'] = choice_set[['walking_time', 'in_transit_time', 'transfer_time']].sum(axis=1)
choice_set

,choice,route,walking_time,in_transit_time,transfer_time,total_time
0,1,U6-U3,1,11,1,13
1,2,U4-U1,1,9,1,11
2,3,U4 + walk,10,7,0,17


### Step 2. Systematic utility $V_j$
The time coefficient is an **assumed, illustrative** value (utility per minute). Its size matters: the Gumbel error has standard deviation $\pi/\sqrt{6} \approx 1.28$ utility units, so $\beta$ sets how many minutes of travel time one unit of randomness is worth.

In [3]:
beta_time = -0.1   # utility per minute (assumed)

choice_set['V'] = beta_time * choice_set['total_time']
choice_set

,choice,route,walking_time,in_transit_time,transfer_time,total_time,V
0,1,U6-U3,1,11,1,13,-1.3
1,2,U4-U1,1,9,1,11,-1.1
2,3,U4 + walk,10,7,0,17,-1.7


### Step 3. Simulate travelers: draw random utility, choose the maximum

In [4]:
num_of_trips = 1000

V = choice_set['V'].to_numpy()
eps = rng.gumbel(loc=0, scale=1, size=(num_of_trips, len(V)))   # random part, one draw per traveler and route
U = V + eps                                                     # total utility
sample_results = choice_set['choice'].to_numpy()[U.argmax(axis=1)]   # each traveler picks the max-utility route
sample_results[:10]

array([2, 2, 3, 2, 3, 3, 2, 1, 2, 1])

In [5]:
for c in choice_set['choice']:
    print(f'Number of travelers choosing option {c} is {np.sum(sample_results == c)}')

Number of travelers choosing option 1 is 339
Number of travelers choosing option 2 is 427
Number of travelers choosing option 3 is 234


### Step 4. Compare with the analytical logit probabilities

In [6]:
expV = np.exp(V)
choice_set['P_logit'] = expV / expV.sum()
choice_set['share_simulated'] = [np.mean(sample_results == c) for c in choice_set['choice']]
choice_set[['choice', 'route', 'total_time', 'V', 'P_logit', 'share_simulated']].round(3)

,choice,route,total_time,V,P_logit,share_simulated
0,1,U6-U3,13,-1.3,0.346,0.339
1,2,U4-U1,11,-1.1,0.422,0.427
2,3,U4 + walk,17,-1.7,0.232,0.234


### Step 5. Sensitivity to the time coefficient
With $\beta = -1/60$ per minute (i.e. utility measured in hours), the routes are almost equally likely; a stronger coefficient makes travelers more deterministic.

In [7]:
for b in [-1/60, -0.05, -0.1, -0.3]:
    e = np.exp(b * choice_set['total_time'])
    print(f'beta = {b:7.4f}/min  ->  P =', (e / e.sum()).round(3).to_list())

beta = -0.0167/min  ->  P = [0.337, 0.348, 0.315]
beta = -0.0500/min  ->  P = [0.342, 0.378, 0.28]
beta = -0.1000/min  ->  P = [0.346, 0.422, 0.232]
beta = -0.3000/min  ->  P = [0.32, 0.583, 0.096]
